In [ ]:
# %matplotlib inline
# !git clone https://github.com/alx87grd/minilink
# import sys
# sys.path.append('/content/minilink')

# Minilink overview

Complete feature lab for minilink — diagrams, custom systems, analysis,
visualization, optimization, compilation, symbolic mechanics, contact engine, and
trajectory optimization.

1. [Quick start](#1.-Quick-start)
2. [Everything is a System](#2.-Everything-is-a-System)
3. [Write your own model](#3.-Write-your-own-model)
4. [Diagram operators](#4.-Diagram-operators)
5. [Manual diagram wiring](#5.-Manual-diagram-wiring)
6. [Custom systems and cascade](#6.-Custom-systems-and-cascade)
7. [Plant catalog and LQR](#7.-Plant-catalog-and-LQR)
8. [Noise ports](#8.-Noise-ports)
9. [Plotting and animation](#9.-Plotting-and-animation)
10. [Interactive mode](#10.-Interactive-mode)
11. [Compilation and JAX benchmarks](#11.-Compilation-and-JAX-benchmarks)
12. [Optimization](#12.-Optimization) — pure NLP (`MathematicalProgram`)
13. [Trajectory optimization (JAX)](#13.-Trajectory-optimization-(JAX))
14. [Symbolic mechanics](#14.-Symbolic-mechanics)
15. [Physics engine](#15.-Physics-engine)
16. [ODE solvers](#16.-ODE-solvers)


In [ ]:
import numpy as np

from minilink.blocks.sources import Step, WhiteNoise
from minilink.control.impedance import ImpedanceController
from minilink.core.diagram import DiagramSystem
from minilink.core.system import DynamicSystem, StaticSystem
from minilink.dynamics.catalog.pendulum.pendulum import Pendulum, PendulumWithNoisePort

## 1. Quick start

One plant, three levels: visualize **f**, simulate, then close the loop with a
controller.

### A plant by itself

Every minilink model is a **`System`**: named input/output **ports**, a state
vector `x`, and parameters `params`. The contract is one **dynamics** equation
plus one map **per output port**:

    dx_dt = f(x, u, t, params)          # how the state evolves
    y     = h(x, u, t, params)          # one equation per output port

Catalog plants and custom subclasses implement `f` and the port output maps in
textbook NumPy style. Pick a plant, set `params` and `x0`, and display the
object — its port diagram shows the boundary signals.


In [ ]:
sys = Pendulum()
sys.params["l"] = 1.2  # arm length [m]
sys.params["m"] = 1.0  # mass [kg]
sys.params["d"] = 0.3  # damping [N.s/m]
sys


Visualize **f**: `plot_phase_plane()` draws the
vector field of the dynamics — how the state would move at each point.


In [ ]:
sys.plot_phase_plane()

In [ ]:
sys.x0[0] = 2.0
sys.compute_trajectory(tf=10.0)
sys.plot_trajectory()
sys.plot_phase_plane()

### Closed loop: `controller @ plant`

The `@` operator builds the standard feedback loop. Chain a step reference with
`>>` into the controller input.


In [ ]:
from minilink.blocks.sources import Step
from minilink.control.impedance import ImpedanceController

step = Step()
step.params["final_value"] = np.array([2.0])
step.params["step_time"] = 5.0

ctl = ImpedanceController()
ctl.params["Kp"] = 100.0
ctl.params["Kd"] = 10.0

closed = step >> ctl @ sys
closed.compute_trajectory(tf=10.0)
closed.plot_trajectory()

## 2. Everything is a System

A `System` owns its equations (`f`, `h`), named **ports**, default
**parameters**, and initial condition. Trajectories live in the returned
`Trajectory` — not hidden inside the object.


In [ ]:
print("Plant:   ", sys.name)
print("states:  ", sys.n, sys.state.labels, sys.state.units)
print("inputs:  ", sys.m, list(sys.inputs))
print("outputs: ", list(sys.outputs))
print("params:  ", sys.params)
print("Diagram: ", closed.n, closed.state.labels, closed.state.units)


## 3. Write your own model

Subclass `DynamicSystem` and implement `f(x, u, t, params)` in textbook style.


In [ ]:
class MassSpringDamper(DynamicSystem):
    # m x'' + c x' + k x = u

    def __init__(self):
        super().__init__(n=2, input_dim=1, expose_state=True)
        self.params = {"m": 1.0, "k": 4.0, "c": 0.3}
        self.state.labels = ["x", "dx"]

    def f(self, x, u, t=0, params=None):
        params = self.params if params is None else params
        m, k, c = params["m"], params["k"], params["c"]
        x, dx = x
        f = u[0]
        ddx = (f - c * dx - k * x) / m
        return np.array([dx, ddx])


msd = MassSpringDamper()
msd.x0[0] = 1.0
msd_chain = Step(final_value=np.array([10.0]), step_time=2.0) >> msd
msd_chain.compute_trajectory(tf=20.0, n_steps=1001)
msd_chain.plot_trajectory()


## 4. Diagram operators


In [ ]:
auto = (Step() + ImpedanceController() + Pendulum()).autowire(strict=True)
auto.plot_diagram()

## 5. Manual diagram wiring


In [ ]:
sys = Pendulum()
sys.params["m"] = 1.0
sys.params["l"] = 5.0
sys.x0[0] = 2.0

step = Step()
step.params["initial_value"] = np.array([0.0])
step.params["final_value"] = np.array([1.0])
step.params["step_time"] = 10.0

ctl = ImpedanceController()
ctl.params["Kp"] = 1000.0
ctl.params["Kd"] = 100.0

diagram = DiagramSystem()
diagram.add_subsystem(step, "step")
diagram.add_subsystem(ctl, "controller")
diagram.add_subsystem(sys, "plant")

diagram.connect("step", "y", "controller", "r")
diagram.connect("controller", "u", "plant", "u")
diagram.connect("plant", "y", "controller", "y")

diagram.plot_diagram()
diagram.compute_trajectory(tf=20)
diagram.plot_trajectory()

In [ ]:
# step.show_signal(t0=0.0, tf=20.0)

## 6. Custom systems and cascade


In [ ]:
class Integrator(DynamicSystem):
    def __init__(self):
        super().__init__(n=1, input_dim=1, output_dim=1, y_dependencies=())
        self.name = "Int"

    def f(self, x, u, t=0, params=None):
        dx = np.zeros(self.n)
        dx[0] = u[0]
        return dx

    def h(self, x, u, t=0, params=None):
        y = np.zeros(self.p)
        y[0] = x[0]
        return y


class PropController(StaticSystem):
    def __init__(self):
        super().__init__()
        self.params = {"Kp": 10.0}
        self.name = "Controller"
        self.add_input_port("r", nominal_value=0.0)
        self.add_input_port("y", nominal_value=0.0)
        self.add_output_port("u", function=self.ctl, dependencies=("r", "y"))

    def ctl(self, x, u, t=0, params=None):
        params = self.params if params is None else params
        r, y = self.get_port_values_from_u(u, "r", "y")
        return np.array([params["Kp"] * (r[0] - y[0])])


sys1 = Integrator()
sys1.state.labels = ["v"]
sys1.x0[0] = 2.0
sys2 = Integrator()
sys2.state.labels = ["x"]
sys2.x0[0] = 2.0

ctl1 = PropController()
ctl1.params["Kp"] = 1.0
ctl2 = PropController()
ctl2.params["Kp"] = 1.0

step = Step()
step.params["initial_value"] = np.array([0.0])
step.params["final_value"] = np.array([1.0])
step.params["step_time"] = 10.0

cascade = DiagramSystem()
cascade.add_subsystem(step, "step")
cascade.add_subsystem(ctl1, "controller1")
cascade.add_subsystem(ctl2, "controller2")
cascade.add_subsystem(sys1, "integrator1")
cascade.add_subsystem(sys2, "integrator2")

cascade.connect("integrator1", "y", "integrator2", "u")
cascade.connect("controller2", "u", "integrator1", "u")
cascade.connect("integrator1", "y", "controller2", "y")
cascade.connect("controller1", "u", "controller2", "r")
cascade.connect("integrator2", "y", "controller1", "y")
cascade.connect("step", "y", "controller1", "r")

cascade

In [ ]:
cascade.compute_trajectory(tf=20)
cascade.plot_trajectory()

## 7. Plant catalog and LQR

`minilink.dynamics.catalog` ships ready-to-use plants; `minilink.control` ships
ready-to-wire controllers and design factories. Below: catalog sweep, forced
cart-pole response, then LQR stabilization.


In [ ]:
from minilink.control.lqr import lqr_at_operating_point
from minilink.dynamics.catalog.aerial.drone import Drone2D
from minilink.dynamics.catalog.pendulum.cartpole import CartPole
from minilink.dynamics.catalog.vehicles.steering import KinematicCar

for plant in (CartPole(), KinematicCar(), Drone2D()):
    print(f"{plant.name:<22} n={plant.n}  m={plant.m}  states={plant.state.labels}")

cartpole = CartPole()
cartpole.compute_forced(lambda t: np.array([3.0 * np.sin(2.0 * t)]))
cartpole.plot_trajectory()


### Control catalog

Controllers are `System` blocks with standard ports (`r`, `y`, `u`).
The quick start used `ImpedanceController`; the catalog also includes
`ImpedanceIntegralController`, `FilteredController`, and LQR design helpers such as
`lqr_at_operating_point` — linearize at an equilibrium and return a
`StateFeedbackController` ready to wire with `@`.


In [ ]:
plant = CartPole()
x_bar = np.array([0.0, np.pi, 0.0, 0.0])
Q = np.diag([1.0, 1.0, 1.0, 1.0])
R = np.array([[1.0]])

lqr_ctl = lqr_at_operating_point(plant, x_bar, Q, R)
lqr_loop = lqr_ctl @ plant

plant.x0 = np.array([-3.0, np.pi - 0.3, 0.0, 0.0])
lqr_loop.compute_trajectory(tf=8.0)
lqr_loop.plot_trajectory()


## 8. Noise ports


In [ ]:
sys = PendulumWithNoisePort()
sys.params["m"] = 1.0
sys.params["l"] = 5.0
sys.x0[0] = 2.0

step = Step()
step.params["initial_value"] = np.array([0.0])
step.params["final_value"] = np.array([1.0])
step.params["step_time"] = 10.0

noise = WhiteNoise(1)
noise.params.update({"var": 1.0, "mean": 0.0, "seed": 1})
noise2 = WhiteNoise(1)
noise2.params.update({"var": 0.1, "mean": 0.0, "seed": 2})

ctl = ImpedanceController()
ctl.params["Kp"] = 1000.0
ctl.params["Kd"] = 100.0

diagram2 = DiagramSystem()
diagram2.add_subsystem(step, "step")
diagram2.add_subsystem(ctl, "controller")
diagram2.add_subsystem(sys, "plant")
diagram2.add_subsystem(noise, "noise")
diagram2.add_subsystem(noise2, "noise2")

diagram2.connect("step", "y", "controller", "r")
diagram2.connect("controller", "u", "plant", "u")
diagram2.connect("plant", "y", "controller", "y")
diagram2.connect("noise", "y", "plant", "w")
diagram2.connect("noise2", "y", "plant", "v")

diagram2

In [ ]:
diagram2.compute_trajectory(solver="euler", dt=0.01)
diagram2.plot_trajectory()

In [ ]:
noise2.show_signal(t0=-300.0, tf=300.0)

## 9. Plotting and animation

Uncomment one renderer at a time — behavior differs in Colab vs local.


In [ ]:
anim_sys = Pendulum()
anim_sys.params["m"] = 1.0
anim_sys.params["l"] = 2.0
anim_sys.x0[0] = 2.0
anim_sys.compute_trajectory(tf=20.0, dt=0.01, show=False)

anim_sys.animate()

In [ ]:
# anim_sys.animate(renderer="pygame")

In [ ]:
anim_sys.animate(renderer="meshcat")

In [ ]:
# anim_sys.animate(renderer="matplotlib", html=True)

## 10. Interactive mode


In [ ]:
# anim_sys.game(renderer="meshcat")

In [ ]:
# anim_sys.game(renderer="pygame")

In [ ]:
from minilink.dynamics.catalog.vehicles.dynamic_bicycle import DynamicBicycleCar3D

car = DynamicBicycleCar3D()
# car.game(renderer="meshcat")

## 11. Compilation and JAX benchmarks

Catalog JAX twins such as `JaxCartPole` compile to a flat execution plan.
`compile(backend="jax")` JIT-compiles **f** and keeps it traceable for
autodiff. The dense-network benchmark below exercises diagram-level compilation.


In [ ]:
import time

import jax
from minilink.core.backends import configure_jax
from minilink.dynamics.catalog.pendulum.cartpole import JaxCartPole

configure_jax(enable_x64=True)

jax_cartpole = JaxCartPole()
jax_eval = jax_cartpole.compile(backend="jax")

n = 1000
xc = np.zeros(jax_cartpole.n)
uc = np.zeros(jax_cartpole.m)

t0 = time.perf_counter()
for _ in range(n):
    jax_cartpole.f(xc, uc, 0.0)
t_interp = time.perf_counter() - t0

t0 = time.perf_counter()
for _ in range(n):
    jax_eval.f(xc, uc, 0.0)
t_jit = time.perf_counter() - t0

print(f"JaxCartPole.f (interpreted) : {1e6 * t_interp / n:6.1f} µs/call")
print(f"compiled (jax jit)          : {1e6 * t_jit / n:6.1f} µs/call  ({t_interp / t_jit:.0f}× faster)")

A = jax.jacfwd(lambda x: jax_eval.f(x, uc, 0.0))(xc)
print("Linearization A = df/dx:\n", np.asarray(A).round(2))


In [ ]:
import time

from minilink.core.system import System

try:
    import jax.numpy as jnp  # noqa: F401

    JAX_AVAILABLE = True
except ImportError:
    JAX_AVAILABLE = False


class SimpleGain(System):
    def __init__(self, id_str, gain=2.0):
        super().__init__(0)
        self.name = id_str
        self.gain = gain
        self.add_input_port("u")
        self.add_output_port("y", function=self.h, dependencies=("u",))

    def h(self, x, u, t=0, params=None):
        return u * self.gain


class SimpleIntegrator(System):
    def __init__(self, id_str):
        super().__init__(1)
        self.name = id_str
        self.add_input_port("u")
        self.add_output_port("x", function=self.compute_state, dependencies=())

    def compute_state(self, x, u, t=0, params=None):
        return x

    def f(self, x, u, t=0, params=None):
        return u * u * u


class MultiInputNode(System):
    def __init__(self, id_str, in_ports):
        super().__init__(1)
        self.name = id_str
        self.in_ports = in_ports
        for p in range(in_ports):
            self.add_input_port(f"u{p}")
        self.add_output_port("x", function=self.compute_state, dependencies="all")

    def compute_state(self, x, u, t=0, params=None):
        return x

    def f(self, x, u, t=0, params=None):
        return u.sum()


def build_dense_network(num_nodes=50, connections_per_node=3):
    diag = DiagramSystem()
    diag.connection_verbose = False
    for i in range(num_nodes):
        diag.add_subsystem(SimpleIntegrator(f"Integrator{i}"), f"Node{i}")

    np.random.seed(42)
    for i in range(1, num_nodes):
        num_conn = min(i, connections_per_node)
        sources = np.random.choice(range(i), size=num_conn, replace=False)
        sys_id = f"MultiNode{i}"
        diag.add_subsystem(MultiInputNode(sys_id, num_conn), sys_id)
        for p_idx, src_i in enumerate(sources):
            diag.connect(f"Node{src_i}", "x", sys_id, f"u{p_idx}")
        diag.connect(sys_id, "x", f"Node{i}", "u")

    diag.add_subsystem(SimpleGain("SourceNode"), "SourceNode")
    diag.connect("SourceNode", "y", "Node0", "u")
    return diag


diag = build_dense_network(num_nodes=10, connections_per_node=10)


diag

In [ ]:
PRINT_COMPILE_REPORT = True
np_eval = diag.compile(backend="numpy", verbose=PRINT_COMPILE_REPORT)
eval_jax = diag.compile(backend="jax", verbose=PRINT_COMPILE_REPORT)

n_iters = 1000
x = np.random.randn(diag.n)
u = np.random.randn(diag.m)

for label, fn in [
    ("Baseline", diag.f),
    ("NumPy compiled", np_eval.f),
    ("JAX JIT", eval_jax.f),
]:
    t0 = time.perf_counter()
    for _ in range(n_iters):
        out = fn(x, u)
        if hasattr(out, "block_until_ready"):
            out.block_until_ready()
    dt = time.perf_counter() - t0
    print(f"{label:16s}: {dt:.4f} s ({n_iters / dt:.0f} evals/sec)")

## 12. Optimization

`minilink.optimization` solves finite-dimensional NLPs: define a
`MathematicalProgram` (`J`, optional constraints), then `Optimizer`.
Trajectory planning (§13) transcribes a `PlanningProblem` into the same layer.

Minimal unconstrained quadratic — min ½‖z − z̄‖², solution z* = z̄:


In [ ]:
import numpy as np

from minilink.optimization.mathematical_program import MathematicalProgram
from minilink.optimization.optimizer import Optimizer

z_bar = np.array([1.0, -0.5, 2.0])

prog = MathematicalProgram(
    n_z=3,
    J=lambda z: 0.5 * np.dot(z - z_bar, z - z_bar),
    grad_J=lambda z: z - z_bar,
)

Optimizer(prog, z0=np.zeros(3), method="scipy_slsqp").solve(disp=True)


## 13. Trajectory optimization (JAX)

Cart-pole swing-up on `jax_cartpole` from §11 — `compile_backend="jax"` reuses
the same JIT-compiled **f** for dynamics defects and NLP gradients.


In [ ]:
from minilink.core.costs import QuadraticCost
from minilink.planning.problems import PlanningProblem
from minilink.planning.trajectory_optimization.direct_collocation import (
    DirectCollocationOptions,
    DirectCollocationTranscription,
)
from minilink.planning.trajectory_optimization.planner import (
    TrajectoryOptimizationOptions,
    TrajectoryOptimizationPlanner,
)

jax_cartpole.inputs["u"].lower_bound[0] = -10.0
jax_cartpole.inputs["u"].upper_bound[0] = 10.0

x_start = np.array([-2.0, 0.0, 0.0, 0.0])
x_goal = np.array([0.0, np.pi, 0.0, 0.0])

problem = PlanningProblem(
    sys=jax_cartpole,
    x_start=x_start,
    x_goal=x_goal,
    cost=QuadraticCost.from_system(
        jax_cartpole, Q=np.diag([1.0, 1.0, 0.0, 0.0]), xbar=x_goal
    ),
)

planner = TrajectoryOptimizationPlanner(
    problem,
    transcription=DirectCollocationTranscription(
        DirectCollocationOptions(tf=4.0, n_steps=20)
    ),
    options=TrajectoryOptimizationOptions(
        compile_backend="jax",
        optimizer_method="ipopt",  # fallback: "scipy_slsqp"
        solve_disp=True,
    ),
)

traj = planner.compute_solution()
planner.plot_solution(signals=("x", "u"))


## 14. Symbolic mechanics


In [ ]:
from minilink.symbolic.mechanics.model import MechanicalModel

m = MechanicalModel("QuadruplePendulum")
N_LINKS = 4

lengths, com_offsets, masses, inertias = [], [], [], []
for i in range(1, N_LINKS + 1):
    li, lci = m.parameters(f"l{i} lc{i}")
    masses.append(m.parameters(f"m{i}"))
    inertias.append(m.parameters(f"I{i}"))
    lengths.append(li)
    com_offsets.append(lci)

g_sym = m.parameters("g")
coords = m.coordinates(" ".join(f"q{i}" for i in range(1, N_LINKS + 1)))
dh_table = [
    {"theta": coords[i], "d": 0, "a": lengths[i], "alpha": 0} for i in range(N_LINKS)
]
link_properties = [
    {"mass": masses[i], "inertia": {"Izz": inertias[i]}, "com_offset": com_offsets[i]}
    for i in range(N_LINKS)
]

m.add_dh_chain(dh_table, link_properties)
m.add_gravity(-g_sym * m.N.y)

print("Deriving symbolic equations of motion...")
sym_sys = m.derive(method="lagrange", simplify=True)
sym_sys.H

In [ ]:
sym_sys.g

In [ ]:
params = {g_sym: 9.81}
for i in range(N_LINKS):
    params[lengths[i]] = 1.5
    params[com_offsets[i]] = 1.5
    params[masses[i]] = 10.0
    params[inertias[i]] = 0.1

sym_plant = sym_sys.to_minilink(parameters=params, backend="jax")
sym_plant.compute_trajectory(tf=10.0)
# sym_plant.animate(renderer="pygame")
sym_plant.animate()

## 15. Physics engine


In [ ]:
from minilink.dynamics.engines.contact_jax import PlaneModel, SphereModel, make_world_model
from minilink.dynamics.engines.world import PhysicsWorldSystem

nx, ny = 12, 10
n_spheres = nx * ny
radii = np.linspace(0.18, 0.35, n_spheres)
masses = np.linspace(0.6, 1.8, n_spheres)
spheres = [SphereModel(mass=m, radius=r) for r, m in zip(radii, masses)]

x_vals = np.linspace(-6.0, 6.0, nx)
y_vals = np.linspace(-4.5, 4.5, ny)
xy_grid = [(x, y) for y in y_vals for x in x_vals]
z_heights = np.linspace(1.0, 6.0, n_spheres)

world = make_world_model(
    spheres,
    PlaneModel(normal=(0.0, 0.0, 1.0), offset=0.0),
    gravity=(0.0, 0.0, -9.81),
    k_contact=1000.0,
    c_contact=1.0,
)

engine_sys = PhysicsWorldSystem(world, name="PhysicsManySpheres")
x0 = np.zeros(engine_sys.n)
for i in range(engine_sys.world.n_bodies):
    base = 13 * i
    x, y = xy_grid[i]
    x0[base : base + 3] = [x, y, z_heights[i]]
    x0[base + 3 : base + 7] = [1.0, 0.0, 0.0, 0.0]
engine_sys.x0 = x0

engine_sys.compute_trajectory(
    tf=10.0, show=False, solver="rk4_fixedsteps", compile_backend="jax"
)
engine_sys.animate(renderer="meshcat")

In [ ]:
import jax

x = np.random.randn(engine_sys.n)
u = np.random.randn(engine_sys.m)
evaluator = engine_sys.compile(backend="jax", verbose=PRINT_COMPILE_REPORT)

n_iter = 1000
t0 = time.perf_counter()
for _ in range(n_iter):
    engine_sys.f(x, u, 0.0)
t_pure = time.perf_counter() - t0

t0 = time.perf_counter()
for _ in range(n_iter):
    evaluator.f(x, u, 0.0)
t_compiled = time.perf_counter() - t0

print(
    f"engine_sys.f:   {1e6 * t_pure / n_iter:.1f} µs/call\n"
    f"compiled (jax): {1e6 * t_compiled / n_iter:.1f} µs/call"
)

df_dx = jax.jacfwd(lambda xx: evaluator.f(xx, u, 0.0))(x)
df_dx.shape

## 16. ODE solvers

Time integration lives in `minilink.simulation`. The central class is
`Simulator`: it compiles the model, builds a time grid, and integrates
`dx/dt = f(x, u, t)` (and `y = h(x, u, t)` when outputs exist). Most
workflows call the façade methods on `System` instead:

- `compute_trajectory(...)` — nominal inputs held at port defaults
- `compute_forced(u, ...)` — prescribed input trajectory or callable

Both accept `solver=...` and `compile_backend=...`. **Solver** picks the
integrator; **compile backend** picks how `f` is evaluated (`numpy` or
`jax` JIT).

If `solver` is omitted, `Simulator.select_solver()` chooses automatically:
`scipy_stiff` for discontinuous models, `rk4_fixedsteps` for long uniform
JAX rollouts (≥ 10 000 points), otherwise `scipy` (RK45).

### Available solver modes

Pass these strings to `solver=` (three underlying backends implement them):

| Mode | Method | Role |
| --- | --- | --- |
| `euler` | explicit Euler | simple fixed-step baseline |
| `rk4_fixedsteps` | classical RK4 | fixed uniform grid; fast with JAX |
| `scipy` | RK45 | default adaptive integrator |
| `scipy_stiff` | Radau | stiff / discontinuous systems |
| `scipy_lsoda` | LSODA | auto stiff / non-stiff switching |
| `scipy_max` | DOP853 | high-accuracy adaptive |
| `scipy_ultra` | DOP853 | tightest tolerances (benchmark truth) |

Example: `sys.compute_trajectory(tf=10.0, solver="euler", dt=0.01)`.

The cell below sweeps solver × compile-backend on a standard pendulum using
the repo-root `benchmarks/` harness (not shipped in the pip package).


In [ ]:
# import sys
# from pathlib import Path

# repo_root = Path.cwd()
# if not (repo_root / "benchmarks").is_dir():
#     repo_root = repo_root.parent.parent
# if str(repo_root) not in sys.path:
#     sys.path.insert(0, str(repo_root))

from benchmarks.simulation import (
    DEFAULT_SIMULATION_VARIANTS,
    benchmark_simulation_matrix,
    print_simulation_matrix_benchmark,
)
from benchmarks.systems.basic import make_pendulum

sys = make_pendulum()

res = benchmark_simulation_matrix(
    sys,
    case_name="Pendulum",
    variants=DEFAULT_SIMULATION_VARIANTS,
    t0=0.0,
    tf=100.0,
    dt=0.01,
    n_runs=5,
)
print_simulation_matrix_benchmark(res)
